# ⚡ cryoDRGN — fast staged downsampling (Colab)

Downsampling a large cryoSPARC stack straight off the Google Drive mount is slow. This notebook
makes it faster by **staging the raw images on local disk in waves**, then downsampling each wave.

### Why this is faster

Reading particles through the Drive FUSE mount is the bottleneck — not the FFT. Two reasons:

1. **Random access.** When a wave's particle indices aren't contiguous within a source `.mrc`,
   cryoDRGN issues **one `seek()` + read per particle**. Over FUSE that's a network round-trip per
   particle — thousands of tiny reads.
2. **No read-ahead.** A bulk `cp` of a whole file is one long sequential transfer, which FUSE
   handles far better, and many files can be copied in parallel.

So we trade *many small random reads over Drive* for *a few large sequential copies*, then read
locally at SSD speed.

### Why waves

The **full-size** raw data is typically far bigger than Colab's local disk (hundreds of GB for a
large stack, vs ~100 GB of disk), so we can't stage it all. Instead we process in waves:

> slice the `.cs` → copy just that wave's `.mrc` files → downsample → delete → next wave

### Correctness

Particle **order is preserved exactly**. Each wave is a contiguous slice of the original `.cs`
(a `.cs` is a `.npy` structured array, so slicing keeps every field), and waves are concatenated
in order. The output therefore matches what `cryodrgn downsample --chunk` would produce, so the
`pose.pkl` / `ctf.pkl` you parse from the **full** `.cs` stay aligned. Downsampling itself is done
by **cryoDRGN's own `downsample` command** — no reimplemented math.

> **Use the main notebook** (`cryoDRGN_colab.ipynb`) for everything else — poses, CTF, training,
> analysis. This notebook only replaces its Step 4.1 for large `.cs` datasets.

## 1 · Setup

In [ ]:
#@title 1.1 · Install cryoDRGN { display-mode: "form" }
#@markdown CPU-only runtime is fine (and cheaper) — downsampling never touches the GPU.
release_channel = "stable"  #@param ["stable", "beta"]
restart_after_install = True  #@param {type:"boolean"}

import subprocess, sys
if release_channel == "beta":
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-i", "https://test.pypi.org/simple/",
           "--extra-index-url", "https://pypi.org/simple/", "cryodrgn", "--pre"]
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "cryodrgn"]
print("Installing cryoDRGN...\n")
if subprocess.run(cmd).returncode != 0:
    raise SystemExit("❌ pip install failed — see the log above.")

# cryoDRGN pins torch<2.10, which can downgrade Colab's torch and leave the pre-installed
# torchvision unable to register its ops. The CLI imports every command module (one of which
# imports umap -> torchvision), so a mismatch breaks *all* cryodrgn commands.
import importlib.metadata as md_

def _v(p):
    try:
        return md_.version(p)
    except md_.PackageNotFoundError:
        return None

tv, tvv = _v("torch"), _v("torchvision")
if tv and tvv:
    tmaj, tmin = (int(x) for x in tv.split(".")[:2])
    if tmaj == 2 and int(tvv.split(".")[1]) != tmin + 15:
        print(f"\n⚠️  torch {tv} / torchvision {tvv} mismatch — realigning...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                        f"torchvision==0.{tmin + 15}.*"])

print("\n✅ Installed.")
if restart_after_install:
    print("🔄 Restarting the runtime (normal) — continue from cell 1.2 afterwards.")
    get_ipython().kernel.do_shutdown(True)

In [ ]:
#@title 1.2 · Mount Drive & set paths { display-mode: "form" }
#@markdown Path to the cryoSPARC **`.cs`** file describing your particles.
cs_file = "/content/drive/MyDrive/mina53_mpp6/J487_particles/J487_particles_exported.cs"  #@param {type:"string"}
#@markdown Folder the `.cs` blob paths resolve against (cryoDRGN's `--datadir`).
datadir = "/content/drive/MyDrive/mina53_mpp6/J486_particles_0"  #@param {type:"string"}
#@markdown Where the finished downsampled stack is saved (durable).
drive_out_dir = "/content/drive/MyDrive/mina53_mpp6/cryodrgn_stack"  #@param {type:"string"}
#@markdown Fast local scratch used for staging + output.
local_dir = "/content/cryodrgn_fast"  #@param {type:"string"}

import os
from google.colab import drive
if not os.path.ismount("/content/drive") and not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

STAGE = os.path.join(local_dir, "stage")
OUT = os.path.join(local_dir, "out")
for d in (local_dir, STAGE, OUT, drive_out_dir):
    os.makedirs(d, exist_ok=True)
for k, v in dict(FD_CS=cs_file, FD_DATADIR=datadir, FD_DRIVE_OUT=drive_out_dir,
                 FD_LOCAL=local_dir, FD_STAGE=STAGE, FD_OUT=OUT).items():
    os.environ[k] = v

import shutil
print(f"{'✅' if os.path.exists(cs_file) else '❌'} .cs      : {cs_file}")
print(f"{'✅' if os.path.isdir(datadir) else '❌'} datadir  : {datadir}")
print(f"   staging  : {STAGE}")
print(f"   output   : {OUT}  →  {drive_out_dir}")
print(f"   local disk free: {shutil.disk_usage(local_dir).free / 1e9:.0f} GB")

## 2 · Inspect the dataset

Reads the `.cs` and works out how many source files there are, how the particles are distributed
across them, and — critically — whether each wave's reads would be **contiguous**. Non-contiguous
reads are what make the Drive mount slow, and the bigger that number, the more staging wins.

In [ ]:
#@title 2.1 · Analyse the .cs { display-mode: "form" }
#@markdown Particles per wave. Each wave's source files must fit on local disk, so the analysis
#@markdown below reports the staged size — lower this if it warns.
wave_size = 20000  #@param {type:"integer"}

import os, numpy as np
CS, DATADIR = os.environ["FD_CS"], os.environ["FD_DATADIR"]

cs = np.load(CS, mmap_mode="r")
N = len(cs)
paths = np.array([p[1:] if p.startswith(">") else p
                  for p in cs["blob/path"].astype(str)])
idxs = np.asarray(cs["blob/idx"])
uniq = np.unique(paths)
print(f"particles       : {N:,}")
print(f"source .mrc     : {len(uniq):,} unique files")

# raw box, dtype and particles-per-file from the first available source file.
# NOTE: never assume float32 — cryoSPARC often writes 2-byte images, and `downsample`
# preserves the source dtype, so this sets both the read volume and the output size.
raw_D = itemsize = None
for rel in uniq[:20]:
    fp = os.path.join(DATADIR, rel)
    if os.path.exists(fp):
        from cryodrgn.mrcfile import MRCHeader
        h = MRCHeader.parse(fp)
        raw_D, per_file = h.D, h.N
        itemsize = np.dtype(h.dtype).itemsize
        print(f"raw box         : {raw_D} x {raw_D}, {np.dtype(h.dtype).name} "
              f"({itemsize} bytes/px)")
        print(f"particles/file  : ~{per_file}  →  ~{-(-N // max(per_file, 1)):,} source files")
        break
if itemsize:
    print(f"raw data to read: {N * raw_D * raw_D * itemsize / 1e9:.0f} GB")
    for d in (128, 256):
        print(f"  output at D={d:<4}: {N * d * d * itemsize / 1e9:.1f} GB")
else:
    print("⚠️  No source .mrc found under datadir — check the path (or wait for the copy/rclone).")

nwaves = -(-N // int(wave_size))
span, noncontig, staged = [], 0, []
for w in range(nwaves):
    sl = slice(w * int(wave_size), min((w + 1) * int(wave_size), N))
    wp, wi = paths[sl], idxs[sl]
    files = np.unique(wp)
    span.append(len(files))
    tot = 0
    for f in files:
        fp = os.path.join(DATADIR, f)
        sel = np.sort(wi[wp == f])
        if not np.all(sel == sel[0] + np.arange(len(sel))):
            noncontig += 1
        try:
            tot += os.path.getsize(fp)
        except OSError:
            pass
    staged.append(tot)

print(f"\nwave size       : {int(wave_size):,} particles  ->  {nwaves} waves")
print(f"files per wave  : min {min(span)}, median {int(np.median(span))}, max {max(span)}")
print(f"staged per wave : max {max(staged) / 1e9:.1f} GB")
print(f"non-contiguous  : {noncontig} of {sum(span)} (file, wave) pairs")
if noncontig:
    print("   ↳ these would be read one particle at a time over Drive — staging helps a lot here.")
else:
    print("   ↳ reads are already sequential; staging still helps, but less dramatically.")

import shutil
free = shutil.disk_usage(os.environ["FD_LOCAL"]).free
out_bytes = N * 0  # filled in by cell 3.1 once box size chosen
if max(staged) > free * 0.5:
    print(f"\n⚠️  Max staged wave ({max(staged) / 1e9:.1f} GB) is large vs {free / 1e9:.0f} GB free "
          f"— lower wave_size.")
os.environ["FD_N"] = str(N)
os.environ["FD_RAWD"] = str(raw_D or 0)
os.environ["FD_ITEMSIZE"] = str(itemsize or 4)
os.environ["FD_WAVE"] = str(int(wave_size))

## 3 · Benchmark

Don't take the speedup on faith — measure it. This times reading the same particles **directly
from Drive** versus **after copying the file locally**, so you can see whether staging is worth it
for *your* data and current Drive throughput.

In [ ]:
#@title 3.1 · Drive vs local read speed { display-mode: "form" }
#@markdown Number of particles to time (kept small — this is only a probe).
probe_particles = 400  #@param {type:"integer"}

import os, time, shutil, numpy as np
from cryodrgn.source import ImageSource
CS, DATADIR, STAGE = os.environ["FD_CS"], os.environ["FD_DATADIR"], os.environ["FD_STAGE"]

cs = np.load(CS, mmap_mode="r")
paths = np.array([p[1:] if p.startswith(">") else p for p in cs["blob/path"].astype(str)])
rel = paths[0]
src_fp = os.path.join(DATADIR, rel)

# cryoSPARC files often hold only ~100 particles, so clamp the probe to what exists
drive_src = ImageSource.from_file(src_fp, lazy=True)
n_in_file = drive_src.n
n = max(1, min(int(probe_particles), n_in_file))
sz = os.path.getsize(src_fp)
print(f"probe file: {rel}")
print(f"  {sz / 1e6:.0f} MB, {n_in_file} particles, {drive_src.D}x{drive_src.D}, "
      f"~{sz / max(n_in_file, 1) / max(drive_src.D ** 2, 1):.1f} bytes/px")
print(f"  timing the first {n} particle(s)\n")

t0 = time.time()
a = drive_src.images(np.arange(n))
t_drive = time.time() - t0
print(f"  read {n} particles from Drive : {t_drive:6.1f} s")

dst = os.path.join(STAGE, rel)
os.makedirs(os.path.dirname(dst), exist_ok=True)
t0 = time.time()
shutil.copy2(src_fp, dst)
t_copy = time.time() - t0
sz = os.path.getsize(dst)
print(f"  bulk copy whole file          : {t_copy:6.1f} s  ({sz / 1e6 / max(t_copy, .01):.0f} MB/s)")

t0 = time.time()
b = ImageSource.from_file(dst, lazy=True).images(np.arange(n))
t_local = time.time() - t0
print(f"  read {n} particles locally     : {t_local:6.1f} s")
assert np.allclose(np.asarray(a), np.asarray(b)), "local copy differs from Drive original!"
print("  ✔ local copy is byte-identical")

# per-file totals: Drive-only vs copy-then-read-all
drive_all = t_drive / n * n_in_file
local_all = t_copy + t_local / n * n_in_file
print(f"\nExtrapolated to this file's {n_in_file:,} particles:")
print(f"  straight from Drive : {drive_all / 60:6.1f} min")
print(f"  stage then read     : {local_all / 60:6.1f} min   ->  {drive_all / max(local_all, .01):.1f}x")
if drive_all > local_all * 1.2:
    print("\n✅ Staging is worth it for this dataset — continue to Step 4.")
else:
    print("\nℹ️  Little gain here; the plain notebook's 4.1 would do fine.")
os.remove(dst)

## 4 · Run the staged downsample

Runs in the **background** and is **resumable** — completed waves are skipped, so if the runtime
drops you can just re-run this cell. Stopping the monitor never kills the job.

In [ ]:
#@title 4.1 · Launch / re-attach { display-mode: "form" }
box_size = 128  #@param [64, 128, 256] {type:"raw"}
#@markdown Images processed at once (RAM ≈ `batch × rawbox² × 20 B`).
batch_size = 1000  #@param {type:"integer"}
#@markdown Parallel file copies during staging — the main throughput knob for Drive.
copy_threads = 8  #@param {type:"integer"}

import os, time, glob, subprocess
from IPython.display import clear_output

L = os.environ["FD_LOCAL"]
worker = os.path.join(L, "stage_downsample.py")
log, pidf, rcf = (os.path.join(L, x) for x in ("job.log", "job.pid", "job.rc"))

script = """
import os, sys, shutil, subprocess, numpy as np
from concurrent.futures import ThreadPoolExecutor

CS, DATADIR = os.environ["FD_CS"], os.environ["FD_DATADIR"]
STAGE, OUT = os.environ["FD_STAGE"], os.environ["FD_OUT"]
D, WAVE, BATCH, THREADS = (int(os.environ[k]) for k in ("FD_D", "FD_WAVE", "FD_BATCH", "FD_THREADS"))

cs = np.load(CS, mmap_mode="r")
N = len(cs)
paths = np.array([p[1:] if p.startswith(">") else p for p in cs["blob/path"].astype(str)])
nwaves = -(-N // WAVE)
print("%d particles, %d waves of %d" % (N, nwaves, WAVE), flush=True)

def files_for(w):
    sl = slice(w * WAVE, min((w + 1) * WAVE, N))
    return sorted(set(paths[sl].tolist()))

def stage(rels):
    def cp(rel):
        s, d = os.path.join(DATADIR, rel), os.path.join(STAGE, rel)
        if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s):
            return 0
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copy2(s, d)
        return os.path.getsize(d)
    with ThreadPoolExecutor(max_workers=THREADS) as ex:
        return sum(ex.map(cp, rels))

for w in range(nwaves):
    out_mrcs = os.path.join(OUT, "particles.%d.%d.mrcs" % (D, w))
    lo, hi = w * WAVE, min((w + 1) * WAVE, N)
    want = (hi - lo) * D * D * 4
    if os.path.exists(out_mrcs) and os.path.getsize(out_mrcs) >= want:
        print("wave %d/%d: already done, skipping" % (w + 1, nwaves), flush=True)
        continue

    rels = files_for(w)
    print("wave %d/%d: staging %d file(s)..." % (w + 1, nwaves, len(rels)), flush=True)
    t0 = __import__("time").time()
    nb = stage(rels)
    print("   staged %.1f GB in %.0fs" % (nb / 1e9, __import__("time").time() - t0), flush=True)

    # a .cs is a .npy structured array: slice it so only this wave's files are referenced
    wave_cs = os.path.join(STAGE, "_wave.cs")
    with open(wave_cs, "wb") as f:
        np.lib.format.write_array(f, np.array(cs[lo:hi]))

    cmd = ['cryodrgn', 'downsample', wave_cs, '-D', str(D), '-o', out_mrcs,
           '-b', str(BATCH), '--datadir', STAGE]
    print("   $ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd)
    if r.returncode != 0:
        print("wave %d FAILED (exit %d)" % (w + 1, r.returncode), flush=True)
        sys.exit(r.returncode)

    keep = set(files_for(w + 1)) if w + 1 < nwaves else set()
    for rel in rels:
        if rel not in keep:
            try:
                os.remove(os.path.join(STAGE, rel))
            except OSError:
                pass
    print("wave %d/%d: done -> %s" % (w + 1, nwaves, out_mrcs), flush=True)

print("ALL WAVES COMPLETE", flush=True)
"""

def live():
    try:
        pid = int(open(pidf).read().strip())
        cl = open("/proc/%d/cmdline" % pid, "rb").read().decode("utf8", "ignore")
    except Exception:
        return None
    return pid if "python" in cl or "stage_downsample" in cl else None

def nbytes():
    return sum(os.path.getsize(f) for f in glob.glob(os.path.join(os.environ["FD_OUT"], "*.mrcs")))

pid = live()
if pid:
    print("↻ Re-attaching to running job (PID %d)" % pid)
else:
    open(worker, "w").write(script)
    os.environ["FD_D"], os.environ["FD_BATCH"] = str(int(box_size)), str(int(batch_size))
    os.environ["FD_THREADS"] = str(int(copy_threads))
    sh = os.path.join(L, "job.sh")
    open(sh, "w").write("#!/bin/bash\necho $$ > %s\npython3 -u %s\necho $? > %s\n"
                        % (pidf, worker, rcf))
    for f in (pidf, rcf):
        if os.path.exists(f):
            os.remove(f)
    subprocess.Popen(["bash", sh], stdout=open(log, "w"), stderr=subprocess.STDOUT,
                     stdin=subprocess.DEVNULL, start_new_session=True, cwd=L,
                     env={**os.environ})
    for _ in range(60):
        pid = live()
        if pid:
            break
        time.sleep(0.25)
    print("🚀 Launched in background (PID %s)" % pid)

expect = (int(os.environ["FD_N"]) * int(box_size) ** 2
          * int(os.environ.get("FD_ITEMSIZE", 4)))  # output keeps the source dtype
t0, b0, interrupted = time.time(), nbytes(), False
try:
    while True:
        alive, nb, el = live(), nbytes(), time.time() - t0
        try:
            tail = [l for l in open(log).read().strip().split("\n") if l][-8:]
        except Exception:
            tail = []
        clear_output(wait=True)
        print("⏳ staged downsample — %s, elapsed %.0f min"
              % (("running (PID %d)" % alive) if alive else "finished", el / 60))
        print("   written %.2f GB / ~%.1f GB  (%.0f%%)" % (nb / 1e9, expect / 1e9, 100.0 * nb / expect))
        rate = (nb - b0) / max(1.0, el)
        if rate > 1e5 and nb < expect:
            print("   %.0f MB/s → ETA ~%.0f min" % (rate / 1e6, (expect - nb) / rate / 60))
        for l in tail:
            print("   " + l[:150])
        print("\n⏹ Stop this cell any time — the job keeps running. Re-run to re-attach.")
        if not alive:
            break
        time.sleep(15)
except KeyboardInterrupt:
    interrupted = True
    print("\n⏹ Detached — job STILL RUNNING. Re-run this cell to re-attach.")

if not interrupted and os.path.exists(rcf):
    rc = int(open(rcf).read().strip() or 1)
    print(("\n✅ All waves complete." if rc == 0 else
           "\n❌ Job failed (exit %d) — see %s" % (rc, log)))

## 5 · Assemble & verify

Writes the `.txt` index cryoDRGN uses to treat the wave files as one stack, checks the particle
count matches the `.cs`, and copies everything to Drive.

In [ ]:
#@title 5.1 · Build the .txt index, verify, and copy to Drive { display-mode: "form" }
copy_to_drive = True  #@param {type:"boolean"}

import os, re, glob, shutil, numpy as np
OUT, DRIVE_OUT = os.environ["FD_OUT"], os.environ["FD_DRIVE_OUT"]
N = int(os.environ["FD_N"])
D = int(re.search(r"particles\.(\d+)\.", os.path.basename(
    sorted(glob.glob(os.path.join(OUT, "particles.*.mrcs")))[0])).group(1))

waves = sorted(glob.glob(os.path.join(OUT, f"particles.{D}.*.mrcs")),
               key=lambda p: int(re.search(r"\.(\d+)\.mrcs$", p).group(1)))
print(f"found {len(waves)} wave file(s)")

from cryodrgn.mrcfile import MRCHeader
total = 0
for w in waves:
    total += MRCHeader.parse(w).N
print(f"particles in output : {total:,}")
print(f"particles in .cs    : {N:,}")
if total != N:
    raise RuntimeError("Particle count mismatch — some waves are missing or incomplete. "
                       "Re-run cell 4.1; it resumes where it left off.")

# .txt lists basenames, resolved relative to the .txt's own directory -> relocatable
txt = os.path.join(OUT, f"particles.{D}.txt")
with open(txt, "w") as f:
    f.write("\n".join(os.path.basename(w) for w in waves))
print(f"✅ index → {txt}")

if copy_to_drive:
    os.makedirs(DRIVE_OUT, exist_ok=True)
    tot = sum(os.path.getsize(w) for w in waves)
    print(f"\n⇪ Copying {tot / 1e9:.1f} GB to {DRIVE_OUT} ...")
    for i, w in enumerate(waves + [txt]):
        dst = os.path.join(DRIVE_OUT, os.path.basename(w))
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(w):
            shutil.copy2(w, dst)
        print(f"   [{i + 1}/{len(waves) + 1}] {os.path.basename(w)}", end="\r")
    print(f"\n✅ Stack on Drive: {os.path.join(DRIVE_OUT, os.path.basename(txt))}")

## 6 · Next steps

Your downsampled stack is now a `particles.<D>.txt` index plus its wave `.mrcs` files. Go back to
the **main notebook** (`cryoDRGN_colab.ipynb`) and:

1. **Skip cell 4.1** — the stack already exists. In cell 2.2 set your Drive project folder to the
   folder holding these files, and 4.1 will find them and restore rather than re-downsample.
   (Make sure `box_size` there matches the `D` you used here.)
2. **Run 4.2 / 4.3** to parse `pose.pkl` and `ctf.pkl` from the **full, original `.cs`** — not the
   per-wave slices. Ordering matches, because waves were concatenated in `.cs` order.
3. Continue with Step 5 onward (sanity check → train → analyze).

**Housekeeping:** the staging folder is emptied as it goes, but a final wave's files may remain —
delete `stage/` if you need the disk space back.